In [104]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [105]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [132]:
basedados_ativos = Path('../../base_dados/brapi/retornos/retornos.csv')
basedados_ibov = Path('../../base_dados/retorno_ibov_2015_2026.csv') 

lista_piotroski = []

for filename in os.listdir(path='../../base_dados/brapi/piotroski/'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/piotroski/', filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))

    df = pd.read_csv(full_path).drop(columns=['Unnamed: 0']).set_index('endDate')
    dicio = {
        'ativo':filename,
        'data':df
    }

    lista_piotroski.append(dicio)

df_ativos=pd.read_csv(basedados_ativos).set_index(['date']).fillna(0)




In [133]:
print(df_ativos.shape)


(2863, 78)


In [134]:
print(f'tamanho mf original: ',len(lista_piotroski))
lista_piotroski_check = []
lista_ativos_finais = []
for i in range(len(lista_piotroski)):
    if lista_piotroski[i]['ativo'] in df_ativos.columns.tolist():
        lista_ativos_finais.append(lista_piotroski[i]['ativo'])
        dc = {
            'ativo':lista_piotroski[i]['ativo'],
            'data':lista_piotroski[i]['data']
        }
        lista_piotroski_check.append(dc)
    else:
        print(f"Nao está na lista: {lista_piotroski[i]['ativo']}")
print(f'tamanho mf Final: ',len(lista_piotroski_check))
        

tamanho mf original:  76
tamanho mf Final:  76


In [135]:
len(df_ativos.columns)

78

In [136]:
df_ativos = df_ativos.filter(lista_ativos_finais)
print(len(df_ativos.columns))

76


### Dados da Magic Formula (SCORE)

In [180]:


# y1 = []
# y2 = []
# for a in anos[:-1]:
#     y1.append(a)
# for a in anos[:-1]:
#     y2.append(a.split('-')[0])

# print("=-"*48)
# print("Anos totais: ",y)
# print("=-"*48)

dict_score_todos = []
dict_score = {}
for i, ativo in enumerate(df_ativos.columns.tolist()):
        if ativo == lista_piotroski_check[i]['ativo']:
            for ano in anos:
                try:
                    df_temp = lista_piotroski_check[i]['data'].query("endDate in @ano")
                    score_anual = (df_temp[ativo]-1)/(8)
                    dict_score[(ano, ativo)] = score_anual.iloc[0]
                    # print("feito dict do ativo: ",ativo,ano,score_anual)
                except:
                     pass
        else:
            print(f" ativo: {ativo} Não tem tamanho suficiente")
            
df_scores = pd.Series(dict_score).unstack()  # index=anos, columns=ativos
df_scores = df_scores.replace([np.inf, -np.inf],0)    
df_scores = df_scores.fillna(0, inplace=True)


# dict_score_todos.append(df_scores)  


In [181]:
score_todos_anos = df_scores

In [182]:
score_todos_anos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
2015-12-31,0.375,0.375,0.125,0.500,0.375,0.625,0.500,0.500,0.500,0.375,...,0.375,0.500,0.375,0.875,0.125,0.125,0.000,0.625,0.625,0.500
2016-12-31,0.375,0.500,0.375,0.500,0.750,0.250,0.375,0.625,0.625,0.375,...,0.125,0.875,0.250,0.375,0.500,0.750,0.000,0.625,0.625,0.500
2017-12-31,0.625,0.625,0.750,0.250,0.625,0.375,0.375,0.500,0.500,0.500,...,0.375,1.000,0.625,0.375,0.625,0.750,0.625,0.875,0.375,0.625
2018-12-31,0.625,0.625,0.375,0.750,0.750,0.500,0.625,0.250,0.250,0.250,...,0.500,1.000,0.500,0.750,0.625,0.750,0.750,0.375,0.625,0.625
2019-12-31,0.625,0.625,0.375,0.375,0.375,0.500,0.500,0.625,0.625,0.375,...,0.375,0.375,0.625,0.750,0.500,0.250,0.250,0.375,0.750,0.500
2020-12-31,0.375,0.375,0.500,0.500,0.500,0.875,0.375,0.375,0.375,0.500,...,0.625,0.250,0.375,0.750,0.625,0.875,0.500,0.375,0.750,0.375
2021-12-31,0.750,0.625,0.500,0.500,0.750,0.625,0.375,0.500,0.500,0.500,...,0.500,0.125,0.750,0.500,0.875,0.625,0.625,0.375,0.375,0.500
2022-12-31,0.625,0.750,0.750,0.625,0.625,0.750,0.625,0.375,0.375,0.375,...,0.500,0.000,0.875,0.625,0.250,0.125,0.375,0.500,0.500,0.625
2023-12-31,0.750,0.500,0.375,0.375,0.625,0.375,0.500,0.125,0.125,0.375,...,0.375,0.625,0.625,0.875,0.250,0.375,0.625,0.750,0.625,0.625
2024-12-31,0.625,0.625,0.875,0.750,0.500,0.750,0.500,0.500,0.500,0.375,...,0.625,0.625,0.375,0.500,0.375,0.250,0.500,0.875,0.500,0.750


In [183]:
index_novo = []
for indexx in score_todos_anos.index:
    novo = indexx.split('-')[0]
    index_novo.append(novo)
print(index_novo)

['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [184]:
score_todos_anos.index=  index_novo

In [185]:
score_todos_anos.to_csv('score_piotroski/piotroski.csv')